In [2]:
import requests
import os
from datetime import datetime, timedelta

def download_gfs(date_str, run, forecast_hour, resolution="0p25", save_path="."):
    """
    Downloads GFS GRIB2 files from Google Cloud Storage.
    
    Args:
        date_str (str): YYYYMMDD (e.g., '20260511')
        run (str): '00', '06', '12', or '18'
        forecast_hour (str): 3-digit hour (e.g., '000', '006')
        resolution (str): '0p25', '0p50', or '1p00'
    """
    # 1. Construct the static public URL
    # Format: gfs.YYYYMMDD/RUN/atmos/gfs.tRUNz.pgrb2.RES.fHR
    base_url = "https://googleapis.com"
    file_path = f"gfs.{date_str}/{run}/atmos/gfs.t{run}z.pgrb2.{resolution}.f{forecast_hour}"
    url = f"{base_url}/{file_path}"
    
    target_file = os.path.join(save_path, f"gfs_{date_str}_{run}_{forecast_hour}.grib2")

    print(f"Connecting to: {url}")
    
    try:
        # 2. Use stream=True to handle large binary files without crashing RAM
        with requests.get(url, stream=True, timeout=30) as r:
            if r.status_code == 404:
                print("Error 404: File not found. Check if the date is too old or the run is still uploading.")
                return
            
            r.raise_for_status()
            
            total_size = int(r.headers.get('content-length', 0))
            
            with open(target_file, 'wb') as f:
                downloaded = 0
                for chunk in r.iter_content(chunk_size=1024 * 1024): # 1MB chunks
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        # Optional: simple progress log
                        if total_size > 0:
                            done = int(50 * downloaded / total_size)
                            print(f"\rProgress: [{'=' * done}{' ' * (50-done)}] {downloaded/1e6:.1f}MB", end="")
                            
        print(f"\nSuccess: Saved to {target_file}")

    except requests.exceptions.RequestException as e:
        print(f"\nAn error occurred: {e}")

if __name__ == "__main__":
    # Example: Download the 00z run for May 11, 2026 (0-hour forecast)
    # GFS data is usually available for the last 30 days on GCS
    DATE = "20260511"
    RUN = "00"
    FHR = "000"
    
    download_gfs(DATE, RUN, FHR)

Connecting to: https://googleapis.com/gfs.20260511/00/atmos/gfs.t00z.pgrb2.0p25.f000
Error 404: File not found. Check if the date is too old or the run is still uploading.
